# ClinVar Processing Notebook

The input variant_summary.txt.gz file was downloaded from the [ClinVar FTP site](https://ftp.ncbi.nlm.nih.gov/pub/clinvar/) on 8/26/2025. The contained SNP-phenotype associations were processed to group related phenotypes and remove irrelevant terms, yielding 17744 associations between 4490 genes and 5719 phenotypes.

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np
import pronto
import re
import scipy.spatial.distance as dist
import seaborn as sns
import sys
from tqdm import tqdm
import scanpy as sc

from sklearn.feature_extraction.text import TfidfVectorizer
import anndata
from collections import OrderedDict

# Bokeh
from bokeh.io import output_notebook
from bokeh.plotting import figure, show, save, output_file
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.palettes import Category20
output_notebook()

from IPython.display import display, HTML, Markdown
sys.setrecursionlimit(100000)

## Load and Pre-Process Data

In [ ]:
clinvar = pd.read_csv('variant_summary.txt.gz', sep='\t', compression='gzip')
clinvar

### PLAU/Alzheimer

In [ ]:
clinvar[(clinvar['GeneSymbol'].str.contains('PLAU')) & (clinvar['PhenotypeList'].str.contains('Alzheimer'))][['GeneSymbol','PhenotypeList', 'ReviewStatus']]# & ((clinvar['ReviewStatus']=='reviewed by expert panel') | (clinvar['ReviewStatus'] == 'criteria provided, multiple submitters, no conflicts'))]

In [ ]:
clinvar[(clinvar['GeneSymbol'].str.contains('PLAU')) & ((clinvar['ReviewStatus']=='reviewed by expert panel') | (clinvar['ReviewStatus'] == 'criteria provided, multiple submitters, no conflicts'))][['GeneSymbol','PhenotypeList', 'ReviewStatus']]

### Filter by Review Status

In [ ]:
clinvar = clinvar[(clinvar['ReviewStatus']=='reviewed by expert panel') | (clinvar['ReviewStatus'] == 'criteria provided, multiple submitters, no conflicts')]
clinvar

In [ ]:
clinvar = clinvar[['GeneSymbol', 'PhenotypeIDS', 'PhenotypeList', 'ClinicalSignificance']]
clinvar = clinvar.replace(0, np.nan)
clinvar = clinvar.dropna()
clinvar = clinvar[['GeneSymbol', 'PhenotypeIDS', 'PhenotypeList']].drop_duplicates()
clinvar

In [ ]:
for p in clinvar[clinvar['PhenotypeList'].str.contains('21\)\(p13')]['PhenotypeList'].map(lambda x: x.split('|')).explode():
    print(p)

In [ ]:
gene_info = pd.read_csv('../../tables/Homo_sapiens.gene_info.gz', sep='\t', compression='gzip')
gene_info = gene_info[(gene_info['#tax_id']==9606) & (gene_info['type_of_gene']=='protein-coding')]
gene_info['Symbol'] = gene_info['Symbol'].str.upper()
symbols = set(gene_info['Symbol'])
geneids = gene_info.set_index('Symbol')['GeneID'].to_dict()
synonyms = gene_info.copy()
synonyms['Synonyms'] = synonyms.apply(lambda x: x['Synonyms'].upper().split('|') + [x['Symbol']], axis=1)
synonyms = synonyms.explode('Synonyms')
synonyms['Synonyms'] = synonyms['Synonyms'].str.upper()
synonyms = synonyms[synonyms['Synonyms']!='-']
synonyms = synonyms.set_index('Synonyms')['Symbol'].to_dict()

In [ ]:
def clean_phenotype(phenotype):
    return phenotype.replace('é','e').replace('Ã©','e').replace('ë','e').replace('ö', 'o').replace('ü','u')

In [ ]:
clinvar_interactions = clinvar.copy()
clinvar_interactions['GeneSymbol'] = clinvar_interactions['GeneSymbol'].str.strip().str.split(';')
clinvar_interactions['PhenotypeIDS'] = clinvar_interactions['PhenotypeIDS'].str.replace('MedGen:C3661900','').str.replace('MedGen:CN169374','').str.strip().str.split('|')
clinvar_interactions['PhenotypeList'] = clinvar_interactions['PhenotypeList'].map(lambda x: re.sub(r'(?<![\()].{3});(?!.{0,8}[\(\)])', '|', x))
clinvar_interactions['PhenotypeList'] = clinvar_interactions['PhenotypeList'].str.strip().str.split('|')
clinvar_interactions = clinvar_interactions.explode('GeneSymbol')
clinvar_interactions = clinvar_interactions.explode('PhenotypeIDS')
clinvar_interactions = clinvar_interactions.explode('PhenotypeList')
clinvar_interactions['PhenotypeList'] = clinvar_interactions['PhenotypeList'].map(clean_phenotype).str.strip()
clinvar_interactions = clinvar_interactions[~clinvar_interactions['PhenotypeList'].isin(['not provided', 'not specified','See cases'])]
clinvar_interactions['GeneSymbol'] = clinvar_interactions['GeneSymbol'].map(synonyms)
clinvar_interactions = clinvar_interactions.dropna().drop_duplicates(subset=['GeneSymbol', 'PhenotypeList']).reset_index(drop=True)
clinvar_interactions

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    mondo = pronto.Ontology('mondo.owl')

In [ ]:
def deduplicate_phenotypes(term):
    if term.lower().endswith(("phenotype", "phenotypes")):
        if term:
            term = term[0].upper() + term[1:]
        return term.strip(',')

    term = term.split(', with')[0].split(' with')[0].split(' WITH')[0].split(' WITH')[0].split(' (with')[0]
    term = term.split(',')[0]
    term = term.split('(')[0]

    term = re.sub(
        r'(,\s*[^,()]*type\s*[A-Za-z\dIVX]*|type\s*[A-Za-z\dIVX]*)',
        '',
        term,
        flags=re.IGNORECASE
    )
    term = re.sub(r'^\w*\w-related disorder$', '', term, re.IGNORECASE)
    term = re.sub(r'^\d+\sconditions$', ', ', term)
    term = re.sub(r',\s*\d+,', ', ', term)
    term = re.sub(r'\s*[A-Z\d]{1,3}$', '', term)
    term = term.replace('Disease','disease')
    term = term.strip().strip(',')
    if term:
        term = term[0].upper() + term[1:]

    return term

In [ ]:
clinvar_interactions['Phenotype'] = clinvar_interactions['PhenotypeList'].map(deduplicate_phenotypes)
clinvar_interactions = clinvar_interactions.drop_duplicates(subset=['GeneSymbol', 'Phenotype']).reset_index(drop=True)
clinvar_interactions

In [ ]:
clinvar_interactions['lower'] = clinvar_interactions['Phenotype'].str.lower()
clinvar_interactions = clinvar_interactions.drop_duplicates(subset=['GeneSymbol', 'lower']).reset_index(drop=True)
clinvar_interactions = clinvar_interactions[~clinvar_interactions['lower'].isin(['','see cases', 'inborn genetic diseases'])]
clinvar_interactions = clinvar_interactions.drop(columns=['lower'])
clinvar_interactions

In [ ]:
phenotype_dicts = clinvar_interactions.set_index('PhenotypeList')['PhenotypeIDS'].apply(lambda x: {idmap.split(':')[0]: ':'.join(idmap.split(':')[1:]).split(';')[0] for idmap in x.split(',')}).to_dict()
def get_phenotype_id(phenotype):
    phendict = phenotype_dicts.get(phenotype, {})
    if 'OMIM' in phendict:
        return phendict['OMIM'], f"http://www.omim.org/entry/{phendict['OMIM']}", 80, 'OMIM'
    elif 'MedGen' in phendict:
        return phendict['MedGen'], f"https://www.ncbi.nlm.nih.gov/medgen/{phendict['MedGen']}", 99 , 'MedGen'
    elif 'Human Phenotype Ontology' in phendict:
        return phendict['Human Phenotype Ontology'], f"https://hpo.jax.org/browse/term/{phendict['Human Phenotype Ontology']}", 12, 'HP'
    elif 'MONDO' in phendict:
        return phendict['MONDO'], f"http://purl.obolibrary.org/obo/{phendict['MONDO'].replace(':','_')}", 116, 'MONDO'
    else:
        return 'NULL', 'NULL', 36, 'ClinVar'

In [ ]:
def get_hierarchical_id(term):
    term_name = term['PhenotypeList']
    target = term['Phenotype']
    term_ids = phenotype_dicts[term_name]

    if target in phenotype_dicts:
        term_ids = phenotype_dicts[target]

    if term_name == target or 'MONDO' not in term_ids:
        return term_ids
    
    parents = mondo.get(term_ids['MONDO']).superclasses(distance=5)
    for parent in parents:
        if parent.name.lower()==target.lower():
            pbenotype_dict = {'MONDO':parent.id}
            xrefs = {xref.id.split(':')[0]:xref.id.split(':')[1] for xref in parent.xrefs}
            if 'MEDGEN' in xrefs:
                pbenotype_dict['MedGen']=f'{xrefs["MEDGEN"]}'
            elif 'HP' in xrefs:
                pbenotype_dict['HP']=f'{xrefs["HP"]}'
            phenotype_dicts[target] = pbenotype_dict
            return pbenotype_dict
    
    return {"":""}

In [ ]:
mondo.get('MONDO:0013342').superclasses(distance=3).to_set()

In [ ]:
clinvar_interactions['PhenotypeID'] = clinvar_interactions.apply(get_hierarchical_id, axis=1)
clinvar_interactions

## Process Data for SQL

### Dataset

In [ ]:
#(id, name, name_without_resource, description, association, gene_set_description, gene_sets_description, attribute_set_description, positive_association, negative_association, is_signed, is_continuous_valued, last_updated, directory, num_page_views, resource_fk, measurement_fk, dataset_group_fk, attribute_type_fk, attribute_group_fk, evidence_type, evidence_group, measurement_bias, attribute_type_plural)
(188, 'ClinVar Gene-Phenotype Associations 2025', 'Gene-Phenotype Associations', 'SNP-phenotype associations curated by ClinVar users from various sources', 'gene-phenotype associations by crowd-sourced curation', 'genes associated with the {0} phenotype from the curated ClinVar Gene-Phenotype Associations 2025 dataset.', 'sets of genes associated with phenotypes from the curated ClinVar Gene-Phenotype Associations 2025 dataset.', 'phenotypes associated with {0} gene from the curated ClinVar Gene-Phenotype Associations 2025 dataset.', 0, 0, '2025-09-05', 'clinvar25', 14037, 10, 20, 1, 15, 3, 'genetic association by literature curation', 'curated literature', 'low throughput, hypothesis driven', 'phenotypes')

### Publication

In [ ]:
#(id, long_citation, short_citation, url, pmid, pubmed_url, first_author_last_name, first_author_initials, journal_abbreviation, year, title, volume, pages)
(177, 'Landrum MJ et al, (2018)  ClinVar: improving access to variant interpretations and supporting evidence. Nucleic Acids Res. 46(D1):D1062–D1067', 'Landrum, Nucleic Acids Res, 2018', 'dx.doi.org/10.1093/nar/gkx1153', 29165669, 'https://ncbi.nlm.nih.gov/pubmed/29165669', 'Landrum', 'MJ', 'Nucleic Acids Res', 2018, 'ClinVar: improving access to variant interpretations and supporting evidence', 46, 'D1062-D1067')

(284, 188, 177),
(285, 188, 83)

### Attributes

In [ ]:
index = 556058
attributes = pd.read_csv('../../tables/attribute.txt', sep='\t')
attributes['name_from_naming_authority'] = attributes['name_from_naming_authority'].astype(str).map(str.lower)
attribute_fks = attributes.set_index('name_from_naming_authority')['id'].to_dict()
for phenotype in clinvar_interactions['Phenotype'].unique():
    if phenotype.lower() not in attribute_fks:
        phenid, phenurl, naming_authority, _ = get_phenotype_id(phenotype)
        print((index, phenotype, phenid, phenurl, naming_authority), end=',\n')
        attribute_fks[phenotype.lower()] = index
        index += 1

### Gene Set

In [ ]:
index = 138900000
gene_sets = clinvar_interactions.copy()
gene_sets['lower'] = gene_sets['Phenotype'].str.lower()
gene_sets = gene_sets.drop_duplicates(subset='lower')
gene_set_fks = {}
for phenotype in gene_sets['Phenotype'].unique():
    phenid, phenurl, _, _ = get_phenotype_id(phenotype)
    print((index, phenotype, phenid, phenurl, 188, 15, attribute_fks[phenotype.lower()]), end=',\n')
    gene_set_fks[phenotype.lower()] = index
    index += 1

### Associations

In [ ]:
genes = pd.read_csv('../../tables/gene.txt',sep='\t')
genes['symbol'] = genes['symbol'].str.upper()
genelist = set(genes['symbol'])
genefks = genes.set_index('symbol')['id'].to_dict()
geneids = gene_info['GeneID'].drop_duplicates()

In [ ]:
index = 92000000

associations = clinvar_interactions.copy()
associations['GeneSymbol'] = associations['GeneSymbol'].apply(lambda x: genefks[x.upper()])
associations['Phenotype'] = associations['Phenotype'].apply(lambda x: gene_set_fks[x.lower()])
associations['Threshold'] = 1
associations = associations.get(['GeneSymbol', 'Phenotype', 'Threshold'])
associations.columns = ['gene_fk', 'gene_set_fk', 'threshold_value']
associations = associations.reset_index(drop=True)
associations.index += index
associations = associations.rename_axis('id')
associations.to_csv('../../harmonizome-update/clinvar25.csv')
associations

## Create Downloads

In [ ]:
output_path = 'downloads'
gene_fk_map = {gene_fk:gene_sym for (gene_sym,gene_fk) in genefks.items()}
gene_set_fk_map = gene_sets.set_index('gene_set_fk')['Phenotype'].to_dict()

edges = associations.copy()
edges['Gene'] = edges['gene_fk'].map(gene_fk_map)
edges['Phenotype'] = edges['gene_set_fk'].map(gene_set_fk_map)
edges = edges[['Gene', 'Phenotype']]
edges

### Binary Matrix

In [ ]:
binarymatrix = pd.crosstab(edges['Gene'], edges['Phenotype']).replace(np.nan, 0)
binarymatrixT = binarymatrix.T
binarymatrix.to_csv(f'{output_path}/gene_attribute_matrix.txt.gz', sep='\t', compression='gzip')
binarymatrix

### Gene Attribute Edge List

In [ ]:
geneids = gene_info.set_index('Symbol')['GeneID'].to_dict()
edgelist = edges.copy()
edgelist['Gene ID'] = edgelist['Gene'].apply(lambda x: geneids[x])
edgelist['PhenotypeID'] = edgelist['Phenotype'].apply(lambda x: get_phenotype_id(x)[3]+'_'+str(get_phenotype_id(x)[0].replace('MONDO:','')))
edgelist['Threshold'] = 1
edgelist = edgelist[['Gene', 'Gene ID', 'Phenotype', 'PhenotypeID', 'Threshold']]
edgelist.columns = ['Gene', 'Gene ID', 'Phenotype', 'Phenotype ID',  'Threshold Value']
edgelist.to_csv(f'{output_path}/gene_attribute_edges.txt.gz', sep='\t', compression='gzip')
edgelist

### Gene List

In [ ]:
geneslist = edgelist[['Gene', 'Gene ID']].drop_duplicates().reset_index(drop=True)
geneslist.to_csv(f'{output_path}/gene_list_terms.txt.gz', sep='\t', compression='gzip')
geneslist

### Attribute List

In [ ]:
attributeslist = edgelist[['Phenotype', 'Phenotype ID']].drop_duplicates().reset_index(drop=True)
attributeslist.to_csv(f'{output_path}/attribute_list_entries.txt.gz', sep='\t', compression='gzip')
attributeslist

### Gene Set Library

In [ ]:
with open(f'{output_path}/gene_set_library_crisp.gmt', 'w') as f:
    arr = binarymatrix.reset_index(drop=True).to_numpy(dtype=np.int_)
    attributes = binarymatrix.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*binarymatrix.index[arr[:, i] == 1]])>= 5:
            print(attributes[i], '', *binarymatrix.index[arr[:, i] == 1], sep='\t', end='\n', file=f)

### Attribute Set Library

In [ ]:
with open(f'{output_path}/attribute_set_library_crisp.gmt', 'w') as f:
    arr = binarymatrixT.reset_index(drop=True).to_numpy(dtype=np.int_)
    genes = binarymatrixT.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*binarymatrixT.index[arr[:, i] == 1]])>= 5:
            print(genes[i], '', *binarymatrixT.index[arr[:, i] == 1], sep='\t', end='\n', file=f)

### Gene Similarity Matrix

In [ ]:
gene_similarity_matrix = dist.pdist(binarymatrix.to_numpy(dtype=np.int_), 'cosine')
gene_similarity_matrix = dist.squareform(gene_similarity_matrix)
gene_similarity_matrix = 1 - gene_similarity_matrix

gene_similarity_matrix = pd.DataFrame(data=gene_similarity_matrix, index=binarymatrix.index, columns=binarymatrix.index)
gene_similarity_matrix.index.name = None
gene_similarity_matrix.columns.name = None
gene_similarity_matrix.to_csv(f'{output_path}/gene_similarity_matrix_cosine.txt.gz', sep='\t', compression='gzip')
gene_similarity_matrix

### Attribute Similarity Matrix

In [ ]:
attribute_similarity_matrix = dist.pdist(binarymatrixT.to_numpy(dtype=np.int_), 'cosine')
attribute_similarity_matrix = dist.squareform(attribute_similarity_matrix)
attribute_similarity_matrix = 1 - attribute_similarity_matrix

attribute_similarity_matrix = pd.DataFrame(data=attribute_similarity_matrix, index=binarymatrixT.index, columns=binarymatrixT.index)
attribute_similarity_matrix.index.name = None
attribute_similarity_matrix.columns.name = None
attribute_similarity_matrix.to_csv(f'{output_path}/attribute_similarity_matrix_cosine.txt.gz', sep='\t', compression='gzip')
attribute_similarity_matrix

### Knowledge Graph Serialization

In [ ]:
nodes = {}
edges = []

for gene in geneslist.index:
    gene = geneslist.loc[gene]
    nodes[int(gene['Gene ID'])] = {
        "type":"gene",
        "properties": {
            "id":int(gene['Gene ID']),
            "label":gene['Gene']
        }}

for pert in attributeslist.index:
    pert = attributeslist.loc[pert]
    nodes[pert['Phenotype ID']] = {
        "type":"phenotype",
        "properties": {
            "label":pert['Phenotype'],
            "id":pert['Phenotype ID']
        }}

for edge in edgelist.index:
    edge = edgelist.loc[edge]
    edges.append({
        "source": int(edge['Gene ID']),
        "relation": "has phenotype",
        "target": edge['Phenotype ID'],
        "properties":{
            "id": str(edge['Gene ID'])+":"+edge['Phenotype ID'],
            "source_label":edge['Gene'],
            "source_id": int(edge['Gene ID']),
            "target_id": edge['Phenotype ID'],
            "target_label":edge['Phenotype'],
            "directed":True,
            "threshold":1
        }})

#### RDF

In [ ]:
with open(output_path+'/kg_serializations/clinvar25.rdf', 'w') as f:
    print('@prefix gene: <https://ncbi.nlm.nih.gov/gene/> .', file=f)
    print('@prefix RO: <http://purl.obolibrary.org/RO_> .', file=f)
    print('@prefix MONDO: <http://purl.obolibrary.org/obo/MONDO_> .', file=f)
    print('@prefix OMIM: <http://www.omim.org/entry/> .', file=f)
    print('@prefix MedGen: <hhttps://www.ncbi.nlm.nih.gov/medgen/> .', file=f)
    print('@prefix HP: <hhttps://hpo.jax.org/browse/term/> .', file=f)
    
    print('', file=f)
    for edge in edges:
        if 'ClinVar' in edge['properties']['target_id']:
            print('gene:'+str(edge['properties']['source_id']),'RO:0002200', f"<ClinVar_{edge['properties']['target_label']}>", end=' .\n', file=f)
        else:
            print('gene:'+str(edge['properties']['source_id']),'RO:0002200', f"{edge['properties']['target_id'].replace('_',':')}", end=' .\n', file=f)

#### JSON

In [ ]:
with open(output_path+'/kg_serializations/clinvar25.json', 'w') as f:
    serial = json.dump(
        {
            "Version":"1", 
            "nodes": nodes,
            "edges": edges
        }, indent=4, fp=f)

#### TSV

In [ ]:
def namespace(phenotype_id):
    if type(phenotype_id)==int:
        return 'NCBI Entrez'
    if 'OMIM' in phenotype_id:
        return 'OMIM'
    elif 'MedGen' in phenotype_id:
        return 'MedGen'
    elif 'HP' in phenotype_id:
        return 'HPO'
    elif 'MONDO' in phenotype_id:
        return 'MONDO'
    return 'ClinVar'

In [ ]:
nodeframe = pd.DataFrame(nodes).T
nodeframe['id'] = nodeframe['properties'].apply(lambda x: x['id'])
nodeframe['label'] = nodeframe['properties'].apply(lambda x: x['label'])
#namespace = {'gene':'NCBI Entrez', 'gene perturbation':'Replogle et al., Cell, 2022'}
nodeframe['namespace'] = nodeframe['id'].map(namespace)
nodeframe = nodeframe.get(['namespace', 'id', 'label']).reset_index(drop=True)
nodeframe.to_csv(output_path+'/kg_serializations/clinvar25_tsv/nodes.tsv', sep='\t')
nodeframe

In [ ]:
edgeframe = pd.DataFrame(edges)
edgeframe['threshold'] = edgeframe['properties'].apply(lambda x: x['threshold'])
edgeframe = edgeframe.get(['source', 'relation', 'target', 'threshold'])
edgeframe.to_csv(output_path+'/kg_serializations/clinvar25_tsv/edges.tsv', sep='\t')
edgeframe

## Prepare Visualizations

### Gene-Attribute Clustermap

In [ ]:
sns.clustermap(binarymatrix, cmap='seismic', center=0, figsize=(10,10), xticklabels=False, yticklabels=False)

### Gene-Gene Similarity Clustermap

In [ ]:
sns.clustermap(gene_similarity_matrix, cmap='seismic', center=0)

### Attribute-Attribute Similarity Clustermap

In [ ]:
sns.clustermap(attribute_similarity_matrix, cmap='seismic', center=0, xticklabels=False, yticklabels=False)

### UMAP

In [ ]:
def load_gmt(file):
    gmt = OrderedDict()
    for line in file:
        term, blank, *geneset = line.strip().split('\t')
        gmt[term] = ' '.join(set(geneset))
    return gmt
libdict = load_gmt(open(f'{output_path}/gene_set_library_crisp.gmt', 'r'))
scatterdir = 'images/'

In [ ]:
def process_scatterplot(libdict, nneighbors=30, mindist=0.1, spread=1.0, maxdf=1.0, mindf=1):
    print("\tTF-IDF vectorizing gene set data...")
    vec = TfidfVectorizer(max_df=maxdf, min_df=mindf)
    X = vec.fit_transform(libdict.values())
    print(X.shape)
    adata = anndata.AnnData(X)
    adata.obs.index = libdict.keys()

    print("\tPerforming Leiden clustering...")
    ### the n_neighbors and min_dist parameters can be altered
    sc.pp.neighbors(adata, n_neighbors=nneighbors, use_rep='X')
    sc.tl.leiden(adata, resolution=1.0)
    sc.tl.umap(adata, min_dist=mindist, spread=spread, random_state=42)

    new_order = adata.obs.sort_values(by='leiden').index.tolist()
    adata = adata[new_order, :]
    adata.obs['leiden'] = 'Cluster ' + adata.obs['leiden'].astype('object')

    df = pd.DataFrame(adata.obsm['X_umap'])
    df.columns = ['x', 'y']

    df['cluster'] = adata.obs['leiden'].values
    df['term'] = adata.obs.index
    df['genes'] = [libdict[l] for l in df['term']]

    return df

In [ ]:
def get_scatter_colors(df):
    clusters = pd.unique(df['cluster']).tolist()
    colors = list(Category20[20])[::2] + list(Category20[20])[1::2]
    color_mapper = {clusters[i]: colors[i % 20] for i in range(len(clusters))}
    return color_mapper

def get_scatterplot(scatterdf):
    df = scatterdf.copy()
    color_mapper = get_scatter_colors(df)
    df['color'] = df['cluster'].apply(lambda x: color_mapper[x])

    hover_emb = HoverTool(name="df", tooltips="""
        <div style="margin: 10">
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Gene Set:</span>
                <span style="font-size: 12px">@gene_set</span>
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Coordinates:</span>
                <span style="font-size: 12px">(@x,@y)</span>
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Cluster:</span>
                <span style="font-size: 12px">@cluster</span>
            </div>
        </div>
    """)
    tools_emb = [hover_emb, 'pan', 'wheel_zoom', 'reset', 'save']

    plot_emb = figure(
        width=1000, 
        height=700, 
        tools=tools_emb
    )

    source = ColumnDataSource(
        data=dict(
            x = df['x'],
            y = df['y'],
            gene_set = df['term'],
            cluster = df['cluster'],
            colors = df['color'],
            label = df['cluster']
        )
    )

    # hide axis labels and grid lines
    plot_emb.xaxis.major_tick_line_color = None
    plot_emb.xaxis.minor_tick_line_color = None
    plot_emb.yaxis.major_tick_line_color = None
    plot_emb.yaxis.minor_tick_line_color = None
    plot_emb.xaxis.major_label_text_font_size = '0pt'
    plot_emb.yaxis.major_label_text_font_size = '0pt' 

    plot_emb.output_backend = "svg"    
    
    plot_emb.title = 'Gene Sets in the ClinVar Gene-Phenotype Associations 2025 Library'
    plot_emb.xaxis.axis_label = "UMAP_1"
    plot_emb.yaxis.axis_label = "UMAP_2"
    plot_emb.xaxis.axis_label_text_font_style = 'normal'
    plot_emb.xaxis.axis_label_text_font_size = '18px'
    plot_emb.yaxis.axis_label_text_font_size = '18px'
    plot_emb.yaxis.axis_label_text_font_style = 'normal'
    plot_emb.title.align = 'center'
    plot_emb.title.text_font_size = '18px'
    
    s = plot_emb.scatter(
        'x', 
        'y', 
        size = 4, 
        source = source, 
        color = 'colors'
    )
    
    return plot_emb

In [ ]:
## defaults: nneighbors=30, mindist=0.1, spread=1.0, maxdf=1.0, mindf=1
scatter_df = process_scatterplot(libdict, nneighbors=30,mindist=0.1
     ,spread=0.75 
     ,maxdf=0.5
     ,mindf=2
)

# Display Scatter Plot
plot = get_scatterplot(scatter_df)
output_notebook()
show(plot)

In [ ]:
output_file(filename=f"{scatterdir}/umap.html", title = 'Gene Sets in ClinVar Gene-Phenotype Associations 2025 Library')
save(plot)